In [ ]:
"""
# Problem Statement

You're an AI engineer at a startup building an email management platform. Your team has developed a new spam classification model, and you need to make a data-driven decision about whether to deploy it to production.

### Context

Your company currently uses Model A (the baseline) in production. Your team has developed Model B, which uses a different architecture. The VP of Engineering wants to know: should we deploy Model B?

You have the following data from your experiments:

**Dataset Information:**

- You collected 10,000 emails for testing
- 3,000 emails are spam (30% spam rate)
- 7,000 emails are legitimate (70% legitimate rate)

**Model A Performance (Current Production Model):**

- True Positives: 2,550 (correctly identified spam)
- False Positives: 350 (legitimate emails marked as spam)
- False Negatives: 450 (spam emails that got through)
- True Negatives: 6,650 (correctly identified legitimate emails)

**Model B Performance (New Model):**

- True Positives: 2,700 (correctly identified spam)
- False Positives: 420 (legitimate emails marked as spam)
- False Negatives: 300 (spam emails that got through)
- True Negatives: 6,580 (correctly identified legitimate emails)

### Your Tasks

**Task 1: Basic Performance Metrics**
Calculate and compare the following metrics for both models:

- Accuracy
- Precision
- Recall
- F1 Score

**Task 2: Statistical Significance**
Determine if the accuracy difference between Model A and Model B is statistically significant. Use a significance level of α = 0.05.

**Task 3: Business Impact Analysis**
Your company processes 1 million emails per day. Calculate:

- How many additional spam emails would Model B catch compared to Model A?
- How many additional legitimate emails would Model B incorrectly flag as spam?
- Is this tradeoff worth it from a business perspective?

**Task 4: Confidence Intervals**
Calculate 95% confidence intervals for the precision of both models. Do the confidence intervals overlap?

**Task 5: A/B Test Design**
Your VP wants to run a live A/B test before fully deploying Model B. Design the test:

- How many users should be in each group to detect a 2% improvement in spam detection rate with 80% power?
- How long should the test run if you process 100,000 emails per day?

**Task 6: Cost-Benefit Analysis**
False positives are worse than false negatives in this context because users get upset when legitimate emails go to spam. Assign costs:

- Each false positive costs $2 (in support tickets and user frustration)
- Each false negative costs $0.10 (minor inconvenience of seeing spam)

Which model has lower expected daily cost on 1 million emails?

**Task 7: Bayesian Thinking**
Using Bayes' Theorem, answer this question: If an email is flagged as spam by Model B, what's the probability it's actually spam?
"""

sample = 10000
spam = 3000
non_spam = 7000

# model a
tp_a = 2550
fp_a = 350
fn_a = 450
tn_a = 6650
# model b
tp_b = 2700
fp_b = 420
fn_b = 300
tn_b = 6580


# task 1
def basic_metrics(tp, fp, tn, fn):
    accuracy = (tp + tn) / (tp + fp + tn + fn)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1_score = 2 * (precision * recall) / (precision + recall)
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
    }


metrics_a = basic_metrics(tp_a, fp_a, tn_a, fn_a)
metrics_b = basic_metrics(tp_b, fp_b, tn_b, fn_b)
print(
    "Model A Metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1 Score: {:.4f}".format(
        *metrics_a.values()
    )
)
print(
    "Model B Metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1 Score: {:.4f}".format(
        *metrics_b.values()
    )
)

# task 2
from statsmodels.stats.proportion import proportions_ztest

count = [tp_a + tn_a, tp_b + tn_b]
nobs = [sample, sample]
stat, pval = proportions_ztest(count, nobs)
alpha = 0.05
if pval < alpha:
    print(
        "The accuracy difference is statistically significant (p-value: {:.4f})".format(
            pval
        )
    )
else:
    print(
        "The accuracy difference is not statistically significant (p-value: {:.4f})".format(
            pval
        )
    )

# task 3
# Your company processes 1 million emails per day. Calculate:
# How many additional spam emails would Model B catch compared to Model A?
n = 1000000
additional_spam_caught = (tp_b - tp_a) * (n / sample)
print("Additional spam emails caught by Model B: {:.0f}".format(additional_spam_caught))
# How many additional legitimate emails would Model B incorrectly flag as spam?
additional_legitimate_flagged = (fp_b - fp_a) * (n / sample)
print(
    "Additional legitimate emails incorrectly flagged by Model B: {:.0f}".format(
        additional_legitimate_flagged
    )
)
# Is this tradeoff worth it from a business perspective?
print("Tradeoff Analysis:")
if additional_spam_caught > additional_legitimate_flagged:
    print("Model B catches more spam than it incorrectly flags legitimate emails.")
else:
    print(
        "Model B incorrectly flags more legitimate emails than the additional spam it catches."
    )



Model A Metrics: Accuracy: 0.9200, Precision: 0.8793, Recall: 0.8500, F1 Score: 0.8644
Model B Metrics: Accuracy: 0.9280, Precision: 0.8654, Recall: 0.9000, F1 Score: 0.8824
The accuracy difference is statistically significant (p-value: 0.0328)
Additional spam emails caught by Model B: 15000
Additional legitimate emails incorrectly flagged by Model B: 7000
Tradeoff Analysis:
Model B catches more spam than it incorrectly flags legitimate emails.


In [13]:
# task 4
from statsmodels.stats.proportion import proportion_confint
ci_a = proportion_confint(tp_a+tn_a, tp_a + fp_a+tn_a+fn_a, alpha=0.05, method='wilson')
ci_b = proportion_confint(tp_b+tn_b, tp_b + fp_b+tn_b+fn_b, alpha=0.05, method='wilson')
print("95% Confidence Interval for Model A Precision: ({:.4f}, {:.4f})".format(ci_a[0], ci_a[1]))
print("95% Confidence Interval for Model B Precision: ({:.4f}, {:.4f})".format(ci_b[0], ci_b[1]))
if ci_a[1] < ci_b[0] or ci_b[1] < ci_a[0]:
    print("The confidence intervals do not overlap.")


95% Confidence Interval for Model A Precision: (0.9145, 0.9252)
95% Confidence Interval for Model B Precision: (0.9228, 0.9329)


In [14]:
#task 5:
power = 0.8
min_detectable_effect = 0.02
alpha = 0.05

effect_size = min_detectable_effect / ((metrics_a['precision'] * (1 - metrics_a['precision'])) ** 0.5)
from statsmodels.stats.power import zt_ind_solve_power

required_n = zt_ind_solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    alternative='two-sided'
)

print("Required sample size per variant:", round(required_n))
print("Total required sample size:", round(required_n * 2))

Required sample size per variant: 4165
Total required sample size: 8329


In [15]:
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import zt_ind_solve_power

baseline_rate = 0.10
mde_absolute = 0.02
new_rate = baseline_rate + mde_absolute

effect_size = proportion_effectsize(baseline_rate, new_rate)

sample_size = zt_ind_solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    alternative='two-sided'
)

print(f"Baseline conversion rate: {baseline_rate:.1%}")
print(f"Minimum detectable effect: {mde_absolute:.1%}")
print(f"Target conversion rate: {new_rate:.1%}")
print(f"Relative improvement: {(mde_absolute/baseline_rate*100):.1f}%")
print(f"Required sample size per variant: {sample_size:.0f}")
print(f"Total users needed: {2*sample_size:.0f}")


Baseline conversion rate: 10.0%
Minimum detectable effect: 2.0%
Target conversion rate: 12.0%
Relative improvement: 20.0%
Required sample size per variant: 3835
Total users needed: 7669


In [16]:
#task 6:
cost_fp = 2.0
cost_fn = 0.10
expected_cost_a = (
    (fp_a * cost_fp + fn_a * cost_fn) * (n / sample)
)
expected_cost_b = (
    (fp_b * cost_fp + fn_b * cost_fn) * (n / sample)
)
print("Expected daily cost for Model A: ${:.2f}".format(expected_cost_a))
print("Expected daily cost for Model B: ${:.2f}".format(expected_cost_b))
if expected_cost_a < expected_cost_b:
    print("Model A has lower expected daily cost.")
else:
    print("Model B has lower expected daily cost.")

Expected daily cost for Model A: $74500.00
Expected daily cost for Model B: $87000.00
Model A has lower expected daily cost.


In [17]:
# task 7:
spam_p = spam / sample
not_spam_p = non_spam / sample

flagged_by_b = (tp_b + fp_b) / sample

p_spam_given_flagged = (tp_b / sample) / flagged_by_b
print("Probability that an email flagged as spam by Model B is actually spam: {:.4f}".format(p_spam_given_flagged))

Probability that an email flagged as spam by Model B is actually spam: 0.8654


In [20]:
#task 8 bayesian

alpha = 1
beta = 1

alpha_b = alpha + tp_b + tn_b
beta_b = beta + fp_b + fn_b

alpha_a = alpha + tp_a + tn_a
beta_a = beta + fp_a + fn_a

from scipy.stats import beta as beta_dist
sample_a = beta_dist.rvs(alpha_a, beta_a, size=10000)
sample_b = beta_dist.rvs(alpha_b, beta_b, size=10000)
prob_b_better = (sample_b > sample_a).mean()
print("Probability that Model B has higher accuracy than Model A: {:.4f}".format(prob_b_better))

Probability that Model B has higher accuracy than Model A: 0.9822


# not sure how to get number of days to carry the test